In [1]:
### Import Libraries
from pathlib import Path
import numpy as np              # ver_1.26
import pandas as pd             # ver_2.1
import sklearn                  # ver_1.7
from utils.ml_utils import * 

In [2]:
### Data Path
current_path = Path.cwd()          
project_root = current_path

data_path = project_root / "data"
image_path = project_root / "images"

# print("# data path:", data_path)

In [3]:
### Load data
df = pd.read_csv(data_path / 'pre_processing_9514.csv', sep=',', encoding='cp949')
print('# data shape : ', df.shape)

# data shape :  (9514, 340)


---

In [4]:
df['TDVmean'] = df.filter(regex='TDV_').mean(axis=1)
df['TDVstd'] = df.filter(regex='TDV_').std(axis=1)
df['PDVmean'] = df.filter(regex='PDV_').mean(axis=1)
df['PDVstd'] = df.filter(regex='PDV_').std(axis=1)

In [5]:
### train_test_split
seed = 0

X = df.copy()
y = df[['Consensus_label', 'Wiggs_label']].copy()
X = X.drop(['Consensus_label','Wiggs_label'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=seed
)

feature_cols = ['AGIS','CIGTS','MD_slope_value','MD','VFI_slope_value','VFI',
                'TDVmean','TDVstd','PDVmean','PDVstd']

X_train = X_train[feature_cols].copy()
X_test = X_test[feature_cols].copy()

In [6]:
models = [
    ("SVM", Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(random_state=seed))
    ])),
    ("Random Forest", Pipeline([
        ("scaler", StandardScaler()), 
        ("clf", RandomForestClassifier(random_state=seed))
    ])),
    ("Logistic Regression", Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(random_state=seed))
    ])),
    ("XGBoost", Pipeline([
        ("scaler", StandardScaler()),
        ("clf", XGBClassifier(random_state=seed))
    ]))
]

In [7]:
progression_name = ['AGIS', 'CIGTS', 'MD', 'VFI', 'TDV', 'PDV']
for num, name in enumerate(progression_name):
    reduced_feature = [x for x in feature_cols if name not in x]
    X_train_new = X_train[reduced_feature].copy()
    X_test_new = X_test[reduced_feature].copy()
    df_ty = fit_eval(models, X_train_new, X_test_new, y_train['Consensus_label'].values, y_test['Consensus_label'].values, "Consensus_label")
    print(f"--- Supplementary Table {num+1} : Removed {name} Feature  ---")
    print(df_ty.iloc[:,1:],'\n')

--- Supplementary Table 1 : Removed AGIS Feature  ---
                 Model  Accuracy  Recall  Specificity  Precision     F1
0                  SVM     0.945   0.821        0.999      0.996  0.900
1        Random Forest     0.953   0.851        0.997      0.992  0.916
2  Logistic Regression     0.945   0.817        1.000      1.000  0.899
3              XGBoost     0.949   0.847        0.992      0.978  0.908 

--- Supplementary Table 2 : Removed CIGTS Feature  ---
                 Model  Accuracy  Recall  Specificity  Precision     F1
0                  SVM     0.948   0.830        0.999      0.996  0.905
1        Random Forest     0.953   0.849        0.998      0.994  0.916
2  Logistic Regression     0.945   0.817        1.000      1.000  0.899
3              XGBoost     0.951   0.849        0.994      0.984  0.911 

--- Supplementary Table 3 : Removed MD Feature  ---
                 Model  Accuracy  Recall  Specificity  Precision     F1
0                  SVM     0.897   0.657   